In [14]:
import requests

r = requests.get(
    "https://putusan3.mahkamahagung.go.id",
    headers={"User-Agent":"Mozilla/5.0"}
)

print(r.status_code)
print(r.url)
print(r.text[:300])

403
https://putusan3.mahkamahagung.go.id/
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal


In [16]:

import os
import time
import random
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://putusan3.mahkamahagung.go.id"
YEARS = [2021, 2022, 2023, 2024, 2025]
TARGET = 35

os.makedirs("data/raw", exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
})


def safe_get(url):
    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()
        time.sleep(random.uniform(1, 2))
        return r.text
    except Exception as e:
        print(f"[ERROR] {url}")
        print(e)
        return None


def get_putusan_links(year, max_pages=5):
    links = []

    for page in range(1, max_pages + 1):
        url = (
            f"{BASE_URL}/direktori/index/kategori/penganiayaan-1"
            f"/tahunjenis/putus/tahun/{year}/page/{page}.html"
        )

        print(f"Scraping: {url}")

        html = safe_get(url)
        if not html:
            continue

        soup = BeautifulSoup(html, "html.parser")

        found = soup.select("a[href*='/putusan/']")

        page_links = []

        for a in found:
            href = a.get("href")
            if not href:
                continue

            if href.startswith("/"):
                href = BASE_URL + href

            page_links.append(href)

        page_links = list(set(page_links))

        print(f"  Page {page}: {len(page_links)} links")

        links.extend(page_links)

        time.sleep(random.uniform(1, 3))

    return list(set(links))


def find_pdf_link(putusan_url):
    html = safe_get(putusan_url)

    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")

    candidates = (
        soup.select("a[href$='.pdf']")
        + soup.select("a[href*='download']")
        + soup.select("a[href*='/pdf/']")
    )

    for a in candidates:
        href = a.get("href")

        if not href:
            continue

        if not href.startswith("http"):
            href = BASE_URL + href

        return href

    return None


def download_pdf(pdf_url, case_id):
    try:
        r = session.get(pdf_url, timeout=60)

        if r.status_code != 200:
            return False

        if b"%PDF" not in r.content[:20]:
            print("  [SKIP] Bukan file PDF")
            return False

        filename = f"data/raw/case_{case_id:03d}.pdf"

        with open(filename, "wb") as f:
            f.write(r.content)

        print(
            f"  [OK] {filename} "
            f"({len(r.content)//1024} KB)"
        )

        return True

    except Exception as e:
        print("  [ERROR]", e)
        return False


# ==========================
# MAIN
# ==========================

all_links = []

for year in YEARS:
    print(f"\n=== Tahun {year} ===")

    links = get_putusan_links(year)

    print(f"Total tahun {year}: {len(links)}")

    all_links.extend([(link, year) for link in links])

print(f"\nTotal link ditemukan: {len(all_links)}")

count = 1

for putusan_url, year in all_links:

    if count > TARGET:
        break

    print(f"\n[{count}/{TARGET}] {putusan_url}")

    pdf_url = find_pdf_link(putusan_url)

    if not pdf_url:
        print("  [SKIP] PDF tidak ditemukan")
        continue

    if download_pdf(pdf_url, count):
        count += 1

    time.sleep(random.uniform(1, 3))

print(f"\nSelesai. {count-1} PDF berhasil diunduh.")



=== Tahun 2021 ===
Scraping: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/penganiayaan-1/tahunjenis/putus/tahun/2021/page/1.html
[ERROR] https://putusan3.mahkamahagung.go.id/direktori/index/kategori/penganiayaan-1/tahunjenis/putus/tahun/2021/page/1.html
403 Client Error: Forbidden for url: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/penganiayaan-1/tahunjenis/putus/tahun/2021/page/1.html
Scraping: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/penganiayaan-1/tahunjenis/putus/tahun/2021/page/2.html
[ERROR] https://putusan3.mahkamahagung.go.id/direktori/index/kategori/penganiayaan-1/tahunjenis/putus/tahun/2021/page/2.html
403 Client Error: Forbidden for url: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/penganiayaan-1/tahunjenis/putus/tahun/2021/page/2.html
Scraping: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/penganiayaan-1/tahunjenis/putus/tahun/2021/page/3.html
[ERROR] https://putusan3.mahkamahagung.go

In [17]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get("https://putusan3.mahkamahagung.go.id")

print("Title awal:", driver.title)

time.sleep(15)

print("Title setelah 15 detik:", driver.title)

input("Tekan ENTER jika website sudah terbuka...")

print(driver.page_source[:500])

driver.quit()

Title awal: Just a moment...
Title setelah 15 detik: Just a moment...
<html lang="en-US" dir="ltr"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src 'none'; script-src 'nonce-fr76ENA0DMy0FYDsmn5DiL' 'unsafe-eval' https://challenges.cloudflare.com; script-src-attr 'none'; style-src 'un


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import requests
import os
import time
import random

BASE_URL = "https://putusan3.mahkamahagung.go.id"
YEARS = [2021, 2022, 2023, 2024, 2025]
TARGET = 35

os.makedirs("data/raw", exist_ok=True)

# ==========================
# BROWSER
# ==========================

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# ==========================
# LOGIN / CAPTCHA MANUAL
# ==========================

print("Membuka website MA...")
driver.get(BASE_URL)

input(
    "\nSelesaikan CAPTCHA sampai website terbuka penuh.\n"
    "Setelah berhasil masuk, tekan ENTER..."
)

print("Judul halaman:", driver.title)

# ==========================
# HELPER
# ==========================

def get_soup(url):
    driver.get(url)
    time.sleep(random.uniform(3, 5))
    return BeautifulSoup(driver.page_source, "html.parser")


def get_putusan_links(year, max_pages=5):
    links = []

    for page in range(1, max_pages + 1):

        url = (
            f"{BASE_URL}/direktori/index/kategori/penganiayaan-1"
            f"/tahunjenis/putus/tahun/{year}/page/{page}.html"
        )

        print(f"\nScraping: {url}")

        soup = get_soup(url)

        found = soup.select("a[href*='/putusan/']")

        page_links = []

        for a in found:
            href = a.get("href")

            if not href:
                continue

            if href.startswith("/"):
                href = BASE_URL + href

            page_links.append(href)

        page_links = list(set(page_links))

        print(f"Page {page}: {len(page_links)} link")

        if len(page_links) == 0:
            break

        links.extend(page_links)

        time.sleep(random.uniform(2, 4))

    return list(set(links))


def get_pdf_link(putusan_url):

    driver.get(putusan_url)
    time.sleep(random.uniform(2, 4))

    soup = BeautifulSoup(driver.page_source, "html.parser")

    pdf_link = (
        soup.select_one("a[href$='.pdf']")
        or soup.select_one("a[href*='download']")
        or soup.select_one("a[href*='/pdf/']")
    )

    if not pdf_link:
        return None

    href = pdf_link["href"]

    if not href.startswith("http"):
        href = BASE_URL + href

    return href


def download_pdf(pdf_url, case_id):

    try:
        cookies = {
            c["name"]: c["value"]
            for c in driver.get_cookies()
        }

        ua = driver.execute_script(
            "return navigator.userAgent;"
        )

        r = requests.get(
            pdf_url,
            cookies=cookies,
            headers={"User-Agent": ua},
            timeout=60
        )

        if r.status_code != 200:
            print("Download gagal:", r.status_code)
            return False

        if b"%PDF" not in r.content[:20]:
            print("Bukan file PDF")
            return False

        filename = f"data/raw/case_{case_id:03d}.pdf"

        with open(filename, "wb") as f:
            f.write(r.content)

        print(
            f"[OK] {filename} "
            f"({len(r.content)//1024} KB)"
        )

        return True

    except Exception as e:
        print("ERROR:", e)
        return False


# ==========================
# AMBIL LINK PUTUSAN
# ==========================

all_links = []

for year in YEARS:

    print(f"\n=== TAHUN {year} ===")

    links = get_putusan_links(year)

    print(f"Total tahun {year}: {len(links)}")

    all_links.extend([(x, year) for x in links])

print("\nTotal link ditemukan:", len(all_links))

# ==========================
# DOWNLOAD PDF
# ==========================

count = 1

for putusan_url, year in all_links:

    if count > TARGET:
        break

    print(
        f"\n[{count}/{TARGET}] "
        f"{putusan_url}"
    )

    pdf_url = get_pdf_link(putusan_url)

    if not pdf_url:
        print("PDF tidak ditemukan")
        continue

    if download_pdf(pdf_url, count):
        count += 1

    time.sleep(random.uniform(2, 5))

driver.quit()

print(
    f"\nSelesai. "
    f"{count - 1} PDF berhasil diunduh."
)

Membuka website MA...
